## Cox con Frailty Compartida — Descripción del modelo

El **Cox con Frailty Compartida (CoxFrailty)** es una extensión del modelo
Cox PH estándar que introduce un término de heterogeneidad aleatoria ω_i
por grupo de observaciones:

    h(t|X, ω_i) = h₀(t) · ω_i · exp(β'X)

donde ω_i sigue una distribución gamma, gaussiana o t compartida entre
todas las ventanas del motor i. El término de frailty captura la
heterogeneidad no observada entre motores — variabilidad en condiciones
de operación, desgaste inicial, o patrones de degradación idiosincrásicos
que las covariables PCA no explican completamente.

**¿Por qué CoxFrailty para RUL?**
El pipeline de ventanas deslizantes genera múltiples filas por motor,
violando el supuesto de independencia del Cox PH estándar. CoxFrailty
es la corrección metodológicamente correcta — el término ω_i agrupa
las ventanas del mismo motor y modela su correlación intra-motor
explícitamente. Es el modelo estadísticamente más riguroso del proyecto
para datos longitudinales con múltiples observaciones por sujeto.

**Abordaje en este proyecto**
Se usa `survival::coxph` con `frailty(motor_id, distribution)` via
la capa `r_repository`. La estimación de θ (varianza de frailty) usa
el algoritmo EM (`method='em'`) o minimización de AIC (`method='aic'`).
`maxit=50` está fijo — valor calibrado empíricamente para viabilidad
computacional en el GGS. RUL se estima via la curva de muerte
F(t|X) = 1 - S(t|X) marginalizando sobre la distribución de frailty.

**Nota computacional**
CoxFrailty es el modelo más costoso del proyecto — ~45 segundos por
configuración con 140 motores y 5 folds. El GGS completo de 3,888
configuraciones requirió ejecución en paralelo (`--jobs 7`).

**Hiperparámetros considerados**

| Hiperparámetro | Rango explorado | Justificación |
|----------------|----------------|---------------|
| `feature_set` | A, B, C, D | Evalúa el impacto del set de features sobre la frailty |
| `window_size` | 20, 25, 30 | Horizonte temporal de cada ventana |
| `n_components` | 10, 15, 20 | Dimensionalidad del espacio de covariables |
| `clipping_threshold` | 115, 120, 125 | Techo del RUL predicho |
| `distribution` | gamma, gaussian, t | Distribución del término de frailty ω_i |
| `method` | em, aic | Método de estimación de la varianza de frailty θ |
| `tdf` | 3, 5 | Grados de libertad de la frailty t — ignorado para gamma/gaussian |
| `confidence_threshold` | 0.3, 0.5, 0.95 | Umbral de F(t) para declarar fallo predicho |

## Cox con Frailty Compartida — Resultados GGS y diagnóstico

**Resumen del GGS**
- Configuraciones evaluadas: 3,888 — exitosas: 3,888 (100%) — fallidas: 0
- Folds: 5 (GroupKFold por motor)
- Tiempo de cómputo: ~45s/config — ejecutado con `--jobs 7`

**Top 10 configuraciones**

| feature_set | window_size | n_components | clipping_threshold | distribution | method | tdf | conf_thresh | S-Score | MAE | RMSE | C-Index |
|-------------|-------------|--------------|-------------------|--------------|--------|-----|-------------|---------|-----|------|---------|
| C | 20 | 20 | 125 | t | em | 5 | 0.3 | 3,390 | 29.4 | 45.5 | 0.569 |
| C | 20 | 20 | 125 | t | aic | 5 | 0.3 | 3,390 | 29.4 | 45.5 | 0.569 |
| C | 20 | 20 | 120 | t | aic | 5 | 0.3 | 3,390 | 31.4 | 45.6 | 0.571 |
| C | 20 | 20 | 120 | t | em | 5 | 0.3 | 3,390 | 31.4 | 45.6 | 0.571 |
| C | 20 | 20 | 115 | t | aic | 5 | 0.3 | 3,391 | 33.6 | 45.9 | 0.573 |
| C | 20 | 20 | 115 | t | em | 5 | 0.3 | 3,391 | 33.6 | 45.9 | 0.573 |
| C | 20 | 20 | 125 | gamma | em | 5 | 0.3 | 3,391 | 29.4 | 45.5 | 0.569 |
| C | 20 | 20 | 125 | gamma | em | 3 | 0.3 | 3,391 | 29.4 | 45.5 | 0.569 |
| C | 20 | 20 | 120 | gamma | em | 5 | 0.3 | 3,391 | 31.4 | 45.6 | 0.571 |
| C | 20 | 20 | 120 | gamma | em | 3 | 0.3 | 3,391 | 31.4 | 45.6 | 0.571 |

### Diagnóstico

**El resultado más llamativo: 100% de convergencia**

A diferencia de CoxPH (22.2% exitosas) y AFT (34.2% exitosas), CoxFrailty
converge en todas las configuraciones. El término de frailty θ actúa como
regularización implícita sobre la heterogeneidad entre motores, estabilizando
numéricamente la estimación de β incluso con 0.4% evento rate. Esto resuelve
el problema numérico pero no el estadístico — β sigue siendo ≈0 porque la
partial likelihood no tiene suficiente información para estimar coeficientes
significativos con tan pocos eventos.

**C-Index ≈ 0.569 — capacidad discriminativa casi nula**

El C-Index de 0.569 es prácticamente aleatorio (0.5 = random) y muy inferior
al de AFT (0.855). La frailty captura variabilidad entre motores pero no mejora
la discriminación entre ventanas en distintos estados de degradación — β≈0
implica que todas las ventanas producen curvas de supervivencia casi idénticas
independientemente del estado del motor.

**Patrones del top 10**

Pipeline: `feature_set=C`, `window_size=20`, `n_components=20` son exclusivos.
Igual que AFT, CoxFrailty necesita features más ricas (memoria + tendencia)
para alimentar el término de frailty — `feature_set=A` no aporta suficiente
variabilidad entre ventanas para estimar θ.

Modelo: `distribution=t` ocupa las primeras 6 posiciones — colas pesadas más
robustas ante heterogeneidad extrema con pocos eventos. `method` es
completamente irrelevante — posiciones 0 y 1 tienen métricas idénticas con
`em` y `aic`, confirmando que el método de estimación de θ no importa cuando
θ no es identificable estadísticamente. `confidence_threshold=0.3` es exclusivo
— mismo patrón que CoxPH y AFT.

**La incompatibilidad estructural persiste**

El término de frailty no resuelve el problema fundamental documentado para
CoxPH y AFT — la tasa de eventos de 0.59% es invariante respecto al modelo:

```
n_eventos / n_filas = 140 / 23,800 ≈ 0.59%  — independiente del modelo
```

CoxFrailty añade un parámetro adicional (θ) que requiere aún más eventos
para identificarse correctamente. Con datos suficientes, la frailty capturaría
heterogeneidad real entre motores y mejoraría las predicciones. Con 0.4%
evento rate, θ no es identificable y el modelo colapsa a CoxPH estándar
con estabilidad numérica mejorada.

### Ranking de modelos de supervivencia

| Métrica | CoxPH | **CoxFrailty** | AFT |
|---------|-------|----------------|-----|
| S-Score | 7,182 | 3,390 | **210** |
| MAE | 32.5 | **29.4** | 38.5 |
| RMSE | 49.4 | **45.5** | 47.2 |
| C-Index | 0.544 | 0.569 | **0.855** |

CoxFrailty es intermedio en S-Score/MAE/RMSE pero su C-Index es similar
a CoxPH — muy por debajo de AFT. AFT sigue siendo el modelo de supervivencia
con mayor capacidad discriminativa real.

### Comparación acumulada completa

| Métrica | NB | DT | RF | SVR | XGB | CoxPH | AFT | **Frailty** |
|---------|----|----|-----|-----|-----|-------|-----|-------------|
| S-Score | 2.447 | 2.755 | 1.883 | 1.825 | 1.782 | 7,182 | 210 | **3,390** |
| MAE | 10.39 | 7.79 | 6.98 | 6.75 | 7.10 | 32.5 | 38.5 | **29.4** |
| RMSE | 13.36 | 12.22 | 10.62 | 10.24 | 10.45 | 49.4 | 47.2 | **45.5** |
| C-Index | 0.908 | 0.893 | 0.909 | 0.915 | 0.912 | 0.544 | 0.855 | **0.569** |

### Conclusión

CoxFrailty aporta un resultado metodológico valioso: demuestra que el término
de frailty estabiliza numéricamente la estimación Cox con datos escasos (100%
convergencia vs 22.2% de CoxPH), pero no resuelve la incompatibilidad
estadística con el 0.4% evento rate. El método de estimación de θ (em vs aic)
es completamente irrelevante — confirmando que θ no es identificable con
los datos disponibles.

**No se seleccionan hiperparámetros de producción.** CoxFrailty queda
documentado como resultado negativo con valor metodológico doble: confirma
la incompatibilidad estructural de los modelos de supervivencia con el
pipeline de ventanas deslizantes en C-MAPSS FD001, y demuestra que añadir
términos de heterogeneidad aleatoria no resuelve el problema cuando la
causa raíz es la escasez de eventos — no la violación del supuesto de
independencia.

In [2]:
import pandas as pd

results_path = 'outputs/ggs/results/CoxFrailty_3da8809d_20260512_2005.csv'

cols_params = ['feature_set', 'window_size', 'n_components', 'clipping_threshold', 
               'distribution', 'method', 'tdf', 'confidence_threshold', 'mean_S_score', 
               'mean_C_index', 'mean_MAE', 'mean_RMSE']

df_results = pd.read_csv(results_path)

total = len(df_results)
exitosas = df_results['mean_S_score'].notna().sum()
fallidas = df_results['mean_S_score'].isna().sum()
n_duplicados = df_results.duplicated(subset=cols_params).sum()

print(f"Total configuraciones: {total}")
print(f"Exitosas:              {exitosas}")
print(f"Fallidas (NaN):        {fallidas}")
print(f"Duplicados:            {n_duplicados}")
print(f"Únicas:                {total - n_duplicados}")

Total configuraciones: 3888
Exitosas:              3888
Fallidas (NaN):        0
Duplicados:            0
Únicas:                3888


In [3]:
df_top = (
    df_results
    .dropna(subset=['mean_S_score'])
    .sort_values('mean_S_score', ascending=True)
    .head(10)
    .reset_index(drop=True)
)

df_top[cols_params]

,feature_set,window_size,n_components,clipping_threshold,distribution,method,tdf,confidence_threshold,mean_S_score,mean_C_index,mean_MAE,mean_RMSE
0,C,20,20,125,t,em,5,0.3,3390.110036,0.568503,29.384668,45.487590
1,C,20,20,125,t,aic,5,0.3,3390.110036,0.568503,29.384668,45.487590
2,C,20,20,120,t,aic,5,0.3,3390.343479,0.570607,31.405497,45.582263
3,C,20,20,120,t,em,5,0.3,3390.343479,0.570607,31.405497,45.582263
4,C,20,20,115,t,aic,5,0.3,3390.787984,0.573023,33.584662,45.927723
5,C,20,20,115,t,em,5,0.3,3390.787984,0.573023,33.584662,45.927723
6,C,20,20,125,gamma,em,5,0.3,3391.062953,0.568667,29.381488,45.478455
7,C,20,20,125,gamma,em,3,0.3,3391.062953,0.568667,29.381488,45.478455
8,C,20,20,120,gamma,em,5,0.3,3391.296397,0.570776,31.402318,45.573180
9,C,20,20,120,gamma,em,3,0.3,3391.296397,0.570776,31.402318,45.573180
